In [22]:
import pandas as pd
import joblib
import re, string
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [23]:
df = pd.read_csv("cleaned_deceptive-opinion.csv")
df.head()

,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,stayed one night getaway family thursday. trip...
1,truthful,hyatt,positive,TripAdvisor,triple rate upgrade view room less $200 also i...
2,truthful,hyatt,positive,TripAdvisor,comes little late finally catching reviews pas...
3,truthful,omni,positive,TripAdvisor,omni chicago really delivers fronts spaciousne...
4,truthful,hyatt,positive,TripAdvisor,asked high floor away elevator got. room pleas...


In [24]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['text'] = df['text'].apply(clean_text)

In [25]:
df['label'] = df['deceptive'].map({'deceptive': 1, 'truthful': 0})

In [26]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [27]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

In [28]:
svm_model = SVC(kernel='linear', C=1.0)
svm_model.fit(X_train_tfidf, y_train)
print("Model training complete")

Model training complete


In [29]:
y_pred = svm_model.predict(X_test_tfidf)
results = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
results.head()

,Actual,Predicted
0,1,1
1,1,1
2,1,1
3,0,0
4,1,1


In [30]:
svm_acc  = accuracy_score(y_test, y_pred)
svm_prec = precision_score(y_test, y_pred)
svm_rec  = recall_score(y_test, y_pred)
svm_f1   = f1_score(y_test, y_pred)
svm_cv   = cross_val_score(svm_model, X_train_tfidf, y_train, cv=5, scoring='accuracy').mean()

print(f"  Accuracy : {svm_acc:.4f}")
print(f"  Precision: {svm_prec:.4f}")
print(f"  Recall   : {svm_rec:.4f}")
print(f"  F1-Score : {svm_f1:.4f}")
print(f"  CV Acc (5-fold): {svm_cv:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Truthful', 'Deceptive']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

  Accuracy : 0.8750
  Precision: 0.8571
  Recall   : 0.9000
  F1-Score : 0.8780
  CV Acc (5-fold): 0.8867

Classification Report:
               precision    recall  f1-score   support

    Truthful       0.89      0.85      0.87       160
   Deceptive       0.86      0.90      0.88       160

    accuracy                           0.88       320
   macro avg       0.88      0.88      0.87       320
weighted avg       0.88      0.88      0.87       320

Confusion Matrix:
[[136  24]
 [ 16 144]]


In [31]:
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(svm_model, 'svm_model.pkl')

print("Model saved as 'svm_model.pkl'")
print("Vectorizer saved as 'tfidf_vectorizer.pkl'")

Model saved as 'svm_model.pkl'
Vectorizer saved as 'tfidf_vectorizer.pkl'
